In [2]:
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch

class SpatialClusterHead(nn.Module):
    def __init__(self, feature_dim, num_clusters):
        super().__init__()
        self.spatial_mlp = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, num_clusters)
        )
        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_clusters)
        )
        
        # THE FIX: Equalizers
        # This prevents the feature_mlp from bullying the spatial_mlp
        self.spatial_norm = nn.LayerNorm(num_clusters)
        self.feature_norm = nn.LayerNorm(num_clusters)
        
    def forward(self, X_combined, spatial_weight=3.0):
        spatial_logits = self.spatial_mlp(X_combined[:, :2])
        feature_logits = self.feature_mlp(X_combined)
        
        # Normalize both to unit variance before combining
        s = spatial_logits / (spatial_logits.std() + 1e-8)
        f = feature_logits / (feature_logits.std() + 1e-8)
        
        
        return spatial_weight * s + f

class FirstTerm(nn.Module):
    def __init__(self, num_cell_types, num_of_clusters , embedding_dim=8):
        super().__init__()
        self.cell_embedding = nn.Embedding(num_cell_types, embedding_dim)
        # Initialize 10.9M weights (The Dials)
        feature_dim = 18 + embedding_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(feature_dim*2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        nn.init.normal_(self.edge_mlp[2].weight, mean=0.0, std=0.001)
        nn.init.constant_(self.edge_mlp[2].bias, 0.01)


        # 3. THE DUAL-PATH CLUSTER HEAD
        # Replaces the standard Sequential MLP
        self.cluster_head = SpatialClusterHead(
            feature_dim=feature_dim, 
            num_clusters=num_of_clusters
        )



    def forward(self, X, X_cell_ids, num_nodes, p_indices, A_skip_csr, current_k, tau=1.0):
        
        X_cell_ids = X_cell_ids.squeeze()
        cell_features = self.cell_embedding(X_cell_ids)  # Shape: [num_nodes, embedding_dim]
        X_combined = torch.cat([X, cell_features], dim=1)  # Shape: [num_nodes, 18 + embedding_dim]

        src_features = X_combined[p_indices[0]]  # Shape: [num_edges, feature_dim]
        dst_features = X_combined[p_indices[1]]  # Shape: [num_edges, feature_dim]
        edge_inputs = torch.cat([src_features, dst_features], dim=1)  # Shape: [num_edges, feature_dim*2]

        dynamic_p_weights = self.edge_mlp(edge_inputs).squeeze(-1)
        safe_weights = F.softplus(dynamic_p_weights) 


        


        # Enforce P >= 0 and build sparse matrix
        P = torch.sparse_coo_tensor(p_indices, safe_weights, 
                                    (num_nodes, num_nodes)).coalesce()
        
        # Reconstruction: XP
        X_hat = torch.sparse.mm(P, X)
        
        # Loss: ||X - XP||
        error = X - X_hat
        loss1 = torch.mean(error**2)   

        # Pass all node features through the head
        logits = self.cluster_head(X_combined) # Shape: [n, k]

        logits = logits[:, :current_k]
        
        #TERM2
        #C matrix with probability distribution across clusters for each node
        C = F.gumbel_softmax(logits, tau=tau, hard=False) # Shape: [n, k]
        # Inside FirstTerm.forward or the training loop
        # tau_smooth = 0.6  # Higher = more spread out/soft
        # C = F.softmax(logits / tau_smooth, dim=-1)
        # C = F.softmax(logits, dim=-1)  # Ensure positivity for SDDMM

        p_vals = P.values()
        
        # 1. Sum across rows (dim=1) to get the total weight leaving each node
        row_sums = torch.sparse.sum(P, dim=1).to_dense()
        
        # 2. Expand row_sums to match the non-zero values 
        # P.indices()[0] contains the row index for every specific edge
        p_vals_norm = p_vals / (row_sums[P.indices()[0]] + 1e-8)
        
        # Rebuild using the exact same sorted indices
        P_norm = torch.sparse_coo_tensor(P.indices(), p_vals_norm, 
                                         (num_nodes, num_nodes)).coalesce()
        
        # 2. Convert to CSR format (Required for the CUDA SDDMM engine)
        P_csr = P_norm.to_sparse_csr()
        
        # 3. SDDMM Magic! 
        # beta=1.0, alpha=-1.0 calculates exactly: (1.0 * P_csr) - (1.0 * C @ C^T)
        # It ONLY calculates this at the 10.9M non-zero locations!
        diff_csr = torch.sparse.sampled_addmm(P_csr, C, C.t(), beta=1.0, alpha=-1.0)
        
        # 4. Square the differences and sum them
        loss2 = torch.sum(diff_csr.values() ** 2)


        #TERM3
        M = torch.matmul(C.t(), torch.sparse.mm(A_skip_csr, C))  # [k, n] @ [n, n] @ [n, k] -> [k, k]
        # 2. Normalize M into a probability distribution (M_tilde)
        M = torch.clamp(M, min=0)
        M_tilde = M / (M.sum() + 1e-8)
        loss3 = -torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        # 3. Calculate Shannon Entropy: -sum(p * log(p))
        # We only calculate for non-zero entries to avoid log(0)
        # loss3 = torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        

        alpha_1 = 1.0 
        alpha_2 = 1.0    
        alpha_3 = 1.0    
        loss = alpha_1* loss1 + (alpha_2 * loss2) +(alpha_3 * loss3) 



        return  loss, loss1 , loss2, loss3,  C , X_combined


import math
def get_tau(epoch):
    tau_start = 2.0
    tau_mid = 1.35   # Targets ~25% Confidence for the middle flatline
    tau_end = 0.85   # Targets ~40% Confidence for the final floor

    if epoch < 75:
        # Phase 1: Smooth descent to the middle flatline
        progress = epoch / 75.0
        return tau_mid + 0.5 * (tau_start - tau_mid) * (1 + math.cos(math.pi * progress))
        
    elif epoch < 150:
        # Phase 2: THE MIDDLE FLATLINE (Epoch 75 to 150)
        return tau_mid
        
    elif epoch < 200:
        # Phase 3: Smooth descent to the final floor
        progress = (epoch - 150) / 50.0
        return tau_end + 0.5 * (tau_mid - tau_end) * (1 + math.cos(math.pi * progress))
        
    else:
        # Phase 4: THE FINAL FLATLINE (Epoch 200 to 250+)
        return tau_end
    


import math

def get_true_compactness_loss(C, positions):
    """
    Dynamically balances tight spatial grouping with 100% cluster utilization.
    Guaranteed to stay positive. Perfect utilization = 0.0 penalty.
    """
    # ==========================================
    # FORCE 1: THE VACUUM (Smooth Spatial Penalty)
    # ==========================================
    weights = C.sum(dim=0) + 1e-8  
    centroids = (C.T @ positions) / weights.unsqueeze(1)  
    
    pos_sq = torch.sum(positions**2, dim=1, keepdim=True)
    cent_sq = torch.sum(centroids**2, dim=1)
    dists_sq = pos_sq + cent_sq - 2 * (positions @ centroids.T)
    
    spatial_loss = (C * (dists_sq ** 2)).sum(dim=1).mean()
    
    # ==========================================
    # FORCE 2: THE WATER PRESSURE (Utilization Reward)
    # ==========================================
    avg_probs = C.mean(dim=0)  # [K]
    K = C.size(1)
    
    # The maximum possible entropy is exactly log(K)
    max_entropy = math.log(K)
    
    # This value is naturally negative
    neg_entropy = torch.sum(avg_probs * torch.log(avg_probs + 1e-10))
    
    # By adding them, perfect balance = 0.0. Complete collapse = +max_entropy.
    utilization_loss = max_entropy + neg_entropy
    
    # ==========================================
    # DYNAMIC BALANCE
    # ==========================================
    # Because utilization_loss is now a smaller positive number (0 to ~6.8),
    # you might need to slightly bump gamma if it doesn't utilize all clusters.
    gamma = 5.0 
    
    return spatial_loss + (gamma * utilization_loss)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F




In [4]:
import os , random
processed_dir = "processed_graphs"
# List all processed file names
design_files = [f for f in os.listdir(processed_dir) if f.endswith('.pt')]

grouped_designs = {}
for f in design_files:
    # Splits "aes_run_20260306..." to extract just "aes"
    base_name = f.split('_run_')[0] 
    if base_name not in grouped_designs:
        grouped_designs[base_name] = []
    grouped_designs[base_name].append(f)

# 2. Pick exactly 20 from each category
UNSEEN_DESIGN = "aes" 

train_files = []
test_files = []

for name, files in grouped_designs.items():
    random.shuffle(files)
    if name == UNSEEN_DESIGN:
        # Truly unseen: keep all AES runs for testing
        test_files.extend(files) 
    else:
        # Standard training: Take 20 balanced samples from others
        selected = files[:20]
        train_files.extend(selected)

random.shuffle(train_files)
random.shuffle(test_files)

# 2. RAM CACHING
train_cache = []
test_cache = []

print(f"📦 Pre-loading TRAIN cache ({len(train_files)} designs)...")
for f in train_files[:20]:
    data = torch.load(os.path.join(processed_dir, f), map_location='cpu')
    train_cache.append((f, data))

print(f"📦 Pre-loading TEST cache ({len(test_files)} designs from {UNSEEN_DESIGN})...")
for f in test_files[:5]:
    data = torch.load(os.path.join(processed_dir, f), map_location='cpu')
    test_cache.append((f, data))

print(f"✅ Pre-loading complete! Training on {len(train_cache)} | Testing on {len(test_cache)}")
 

📦 Pre-loading TRAIN cache (60 designs)...


/home/rain/CTS-Task-Aware-Clustering/venv/lib/python3.12/site-packages/torch/_utils.py:361: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  result = torch.sparse_compressed_tensor(


📦 Pre-loading TEST cache (31 designs from aes)...
✅ Pre-loading complete! Training on 20 | Testing on 5


In [5]:
import torch
import os

# 1. Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load global metadata (needed for model initialization)
metadata = torch.load("global_metadata.pt")
global_max_cell_types = metadata['global_max_cell_types']
global_max_k = metadata['global_max_k']

# 3. Initialize the Phase 1 Model
# (Assuming your FirstTerm and SpatialClusterHead classes are defined above this)
phase1_model = FirstTerm(
    num_cell_types=global_max_cell_types, 
    num_of_clusters=global_max_k
).to(device)

# 4. Load the trained weights
phase1_model.load_state_dict(torch.load("phase1_clustering_ep250_twohop.pt", map_location=device))

# 5. 🔴 CRITICAL: Freeze the model
phase1_model.eval() # Disables dropout and locks batchnorm stats
for param in phase1_model.parameters():
    param.requires_grad = False

print("✅ Phase 1 Model loaded and frozen. Ready to extract C matrices.")

✅ Phase 1 Model loaded and frozen. Ready to extract C matrices.


In [6]:
import torch.nn as nn
import torch.nn.functional as F

class TaskPredictor(nn.Module):
    def __init__(self, feat_dim, num_knobs=4, hidden=128):
        super().__init__()
        # Shared context for the tool knobs
        self.knob_proj = nn.Linear(num_knobs, 32)
        
        # 1. SKEW HEAD: Max-pooling (Outlier-focused)
        self.skew_mlp = nn.Sequential(
            nn.Linear(feat_dim + 32, hidden//2),
            nn.ReLU(),
            nn.Linear(hidden//2, 1)
        )
        
        # 2. POWER HEAD: Mean-pooling (Bulk-focused)
        self.power_mlp = nn.Sequential(
            nn.Linear(feat_dim + 32, hidden//2),
            nn.ReLU(),
            nn.Linear(hidden//2, 1)
        )
        
        # 3. WL HEAD: Mean pooling (Spread-focused)
        self.wl_mlp = nn.Sequential(
            nn.Linear(feat_dim + 32, hidden//2), 
            nn.ReLU(),
            nn.Linear(hidden//2, 1)
        )

    def forward(self, X_tilde, cts_knobs):
        # A. Process Knobs
        k_feat = F.relu(self.knob_proj(cts_knobs)) # [1, 32]
        
        # B. Pool Supernodes (The "Simple" way)
        # Skew Head gets the MAX (worst case)
        h_max = torch.mean(X_tilde, dim=0, keepdim=True) # [1, feat_dim]
        skew_in = torch.cat([h_max, k_feat], dim=-1)    
        
        # Power Head gets the MEAN (average load)
        h_mean = torch.mean(X_tilde, dim=0, keepdim=True) # [1, feat_dim]
        power_in = torch.cat([h_mean, k_feat], dim=-1)
        
        # WL Head gets MEAN 
        wl_in = torch.cat([h_mean, k_feat], dim=-1)
        
        # C. Final Predictions
        return self.skew_mlp(skew_in), self.power_mlp(power_in), self.wl_mlp(wl_in)

In [ ]:
import time
from helper import relative_masking , get_compressed_graph , load_cts_parameters

# Assuming device, phase1_model, and ram_cache are already set up from above

NUM_EPOCHS = 100
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
csv_path = "./dataset_with_def/unified_manifest_normalized.csv"


print("\n🚀 Starting Fast-Track Training Loop (Phase 1 Frozen)...\n")

predictor = TaskPredictor(feat_dim=28).to(device)
optimizer = torch.optim.Adam(predictor.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(NUM_EPOCHS):
    predictor.train()
    t_stats = {'mse': 0, 'mae': 0, 'mae_s': 0, 'mae_p': 0, 'mae_w': 0, 'cnt': 0}
    random.shuffle(train_cache)

    for step, (filename, data) in enumerate(train_cache):
        placement_id = filename.replace('.pt', '')

        with torch.no_grad():
            X = data['X'].to(device)
            X_cell_ids = data['X_cell_ids'].to(device)
            p_indices = data['p_indices'].to(device)
            A_skip_csr = data['A_skip_csr'].to(device)
            A_wire_csr = data['A_wire_csr'].to(device)

            _, _, _, _, C, X_combined = phase1_model(
                X, X_cell_ids, data['num_nodes'],
                p_indices, A_skip_csr, data['current_k'], tau=0.85
            )
            X_tilde, A_tilde_skip, A_tilde_wire = get_compressed_graph(
                X_combined, C, A_skip_csr, A_wire_csr, data['raw_areas'].to(device)
            )

        cts_configs = load_cts_parameters(csv_path, placement_id, device)

        for run in cts_configs:
            knobs = run['knobs'].unsqueeze(0).to(device)
            targets = run['targets']

            s_pred, p_pred, w_pred = predictor(X_tilde, knobs)

            preds = torch.cat([s_pred.view(-1), p_pred.view(-1), w_pred.view(-1)])
            trues = torch.cat([targets['skew'].view(-1), 
                               targets['power'].view(-1), 
                               targets['wl'].view(-1)])

            loss = criterion(preds, trues)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(predictor.parameters(), max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                t_stats['mse'] += loss.item()
                t_stats['mae'] += torch.abs(preds - trues).mean().item()
                t_stats['mae_s'] += torch.abs(s_pred.view(-1) - targets['skew'].view(-1)).item()
                t_stats['mae_p'] += torch.abs(p_pred.view(-1) - targets['power'].view(-1)).item()
                t_stats['mae_w'] += torch.abs(w_pred.view(-1) - targets['wl'].view(-1)).item()
                t_stats['cnt'] += 1

        del X, X_cell_ids, p_indices, A_skip_csr, A_wire_csr
        del C, X_combined, X_tilde, A_tilde_skip, A_tilde_wire
        if step % 5 == 0:
            torch.cuda.empty_cache()

    # --- VALIDATION ---
    predictor.eval()
    v_stats = {'mse': 0, 'mae': 0, 'mae_s': 0, 'mae_p': 0, 'mae_w': 0, 'cnt': 0}

    with torch.no_grad():
        for filename, data in test_cache:
            placement_id = filename.replace('.pt', '')
            X = data['X'].to(device)
            X_cell_ids = data['X_cell_ids'].to(device)
            p_indices = data['p_indices'].to(device)
            A_skip_csr = data['A_skip_csr'].to(device)
            A_wire_csr = data['A_wire_csr'].to(device)

            _, _, _, _, C, X_combined = phase1_model(
                X, X_cell_ids, data['num_nodes'],
                p_indices, A_skip_csr, data['current_k'], tau=0.85
            )
            X_tilde, A_tilde_skip, A_tilde_wire = get_compressed_graph(
                X_combined, C, A_skip_csr, A_wire_csr, data['raw_areas'].to(device)
            )

            cts_configs = load_cts_parameters(csv_path, placement_id, device)
            for run in cts_configs:
                knobs = run['knobs'].unsqueeze(0).to(device)
                targets = run['targets']

                s_p, p_p, w_p = predictor(X_tilde,  knobs)
                val_preds = torch.cat([s_p.view(-1), p_p.view(-1), w_p.view(-1)])
                val_trues = torch.cat([targets['skew'].view(-1),
                                       targets['power'].view(-1),
                                       targets['wl'].view(-1)])
                v_loss = criterion(val_preds, val_trues)

                v_stats['mse'] += v_loss.item()
                v_stats['mae'] += torch.abs(val_preds - val_trues).mean().item()
                v_stats['mae_s'] += torch.abs(s_p.view(-1) - targets['skew'].view(-1)).item()
                v_stats['mae_p'] += torch.abs(p_p.view(-1) - targets['power'].view(-1)).item()
                v_stats['mae_w'] += torch.abs(w_p.view(-1) - targets['wl'].view(-1)).item()
                v_stats['cnt'] += 1

            del X, X_cell_ids, p_indices, A_skip_csr, A_wire_csr
            del C, X_combined, X_tilde, A_tilde_skip, A_tilde_wire

    def avg(stats, key): 
        return stats[key] / stats['cnt'] if stats['cnt'] > 0 else 0

    print(f"\n📊 [EPOCH {epoch+1:03d}]")
    print(f"{'Metric':<12} | {'Train':<12} | {'Val (unseen)':<12}")
    print("-" * 42)
    print(f"{'MSE':<12} | {avg(t_stats,'mse'):<12.6f} | {avg(v_stats,'mse'):<12.6f}")
    print(f"{'MAE':<12} | {avg(t_stats,'mae'):<12.5f} | {avg(v_stats,'mae'):<12.5f}")
    print(f"{'Skew MAE':<12} | {avg(t_stats,'mae_s'):<12.5f} | {avg(v_stats,'mae_s'):<12.5f}")
    print(f"{'Power MAE':<12} | {avg(t_stats,'mae_p'):<12.5f} | {avg(v_stats,'mae_p'):<12.5f}")
    print(f"{'WL MAE':<12} | {avg(t_stats,'mae_w'):<12.5f} | {avg(v_stats,'mae_w'):<12.5f}")
    print("=" * 42)

torch.save(predictor.state_dict(), "phase2_predictor.pt")
print("✅ Saved phase2_predictor.pt")
    



print("\n🎉 Frozen extraction loop verified!")


🚀 Starting Fast-Track Training Loop (Phase 1 Frozen)...


📊 [EPOCH 001]
Metric       | Train        | Val (unseen)
------------------------------------------
MSE          | 1.093826     | 1.109868    
MAE          | 0.86834      | 0.86359     
Skew MAE     | 0.69606      | 0.78196     
Power MAE    | 0.91328      | 0.81290     
WL MAE       | 0.99567      | 0.99589     

📊 [EPOCH 002]
Metric       | Train        | Val (unseen)
------------------------------------------
MSE          | 0.930945     | 0.987020    
MAE          | 0.74159      | 0.80760     
Skew MAE     | 0.64668      | 0.59306     
Power MAE    | 0.76776      | 0.94575     
WL MAE       | 0.81034      | 0.88401     

📊 [EPOCH 003]
Metric       | Train        | Val (unseen)
------------------------------------------
MSE          | 0.750781     | 1.608714    
MAE          | 0.70095      | 1.05990     
Skew MAE     | 0.65338      | 0.60191     
Power MAE    | 0.66028      | 1.39172     
WL MAE       | 0.78920      | 1.18609

In [ ]:
# ==========================================
# 3. Final Evaluation & True vs. Predicted
# ==========================================
print("\n" + "="*50)
print("🚀 FINAL EVALUATION: TRUE VS PREDICTED")
print("="*50)

predictor_model.eval()

mae_skew_sum, mae_power_sum, mae_wl_sum = 0, 0, 0
eval_preds = 0
print_limit = 15 # Only print the first 15 samples to keep the console clean

with torch.no_grad():
    for filename, data in ram_cache:
        placement_id = filename.replace('.pt', '')
        cts_runs = load_cts_parameters(csv_file, placement_id, device)
        if not cts_runs: 
            continue

        # 1. Get Clustering Matrix (Frozen)
        _, _, _, _, C, X_combined = phase1_model(
            data['X'].to(device), data['X_cell_ids'].to(device), 
            data['num_nodes'], data['p_indices'].to(device), 
            data['A_skip_csr'].to(device), data['current_k'], tau=0.85
        )

        # 2. Evaluate CTS Runs
        for run in cts_runs:
            cts_params = run['knobs'].view(1, -1)
            t_p = run['targets']['power'].view(1, 1)
            t_s = run['targets']['skew'].view(1, 1)
            t_w = run['targets']['wl'].view(1, 1)
            
            # Predict
            skew_pred, power_pred, wl_pred = predictor_model(C, X_combined, cts_params)
            
            # Calculate Absolute Errors
            mae_skew_sum += abs(skew_pred.item() - t_s.item())
            mae_power_sum += abs(power_pred.item() - t_p.item())
            mae_wl_sum += abs(wl_pred.item() - t_w.item())
            
            # Print sample True vs Predicted
            if eval_preds < print_limit:
                print(f"Sample {eval_preds+1:2} | "
                      f"SKEW [True: {t_s.item():>6.3f} | Pred: {skew_pred.item():>6.3f}]  ---  "
                      f"POWER [True: {t_p.item():>6.3f} | Pred: {power_pred.item():>6.3f}]")
                
            eval_preds += 1
            
        del C, X_combined
        torch.cuda.empty_cache()

# 3. Calculate and Print Global MAE
final_mae_skew = mae_skew_sum / max(eval_preds, 1)
final_mae_power = mae_power_sum / max(eval_preds, 1)
final_mae_wl = mae_wl_sum / max(eval_preds, 1)

print("-" * 50)
print(f"🎯 FINAL GLOBAL MAE (Across {eval_preds} CTS Runs)")
print(f"   ➤ Skew MAE:  {final_mae_skew:.4f}")
print(f"   ➤ Power MAE: {final_mae_power:.4f}")
print(f"   ➤ WL MAE:    {final_mae_wl:.4f}")
print("="*50)

# Save the predictor once it's trained and evaluated
torch.save(predictor_model.state_dict(), "phase2_predictor.pt")
print("✅ Saved phase2_predictor.pt")


🚀 FINAL EVALUATION: TRUE VS PREDICTED
Sample  1 | SKEW [True: -0.245 | Pred: -0.496]  ---  POWER [True: -1.322 | Pred: -1.433]
Sample  2 | SKEW [True:  1.612 | Pred:  1.427]  ---  POWER [True: -1.237 | Pred: -1.334]
Sample  3 | SKEW [True:  1.392 | Pred:  1.356]  ---  POWER [True: -1.335 | Pred: -1.375]
Sample  4 | SKEW [True: -0.320 | Pred: -0.287]  ---  POWER [True: -1.159 | Pred: -1.120]
Sample  5 | SKEW [True: -0.158 | Pred: -0.272]  ---  POWER [True: -1.236 | Pred: -1.282]
Sample  6 | SKEW [True:  0.981 | Pred:  1.120]  ---  POWER [True: -1.375 | Pred: -1.430]
Sample  7 | SKEW [True: -0.440 | Pred: -0.412]  ---  POWER [True: -1.244 | Pred: -1.289]
Sample  8 | SKEW [True: -0.221 | Pred: -0.418]  ---  POWER [True: -1.239 | Pred: -1.164]
Sample  9 | SKEW [True:  2.027 | Pred:  2.220]  ---  POWER [True: -1.331 | Pred: -1.388]
Sample 10 | SKEW [True: -1.149 | Pred: -0.995]  ---  POWER [True: -1.356 | Pred: -1.324]
Sample 11 | SKEW [True: -0.198 | Pred: -0.368]  ---  POWER [True:  0.78

In [ ]:
import os
import torch
from helper import load_cts_parameters

# ==========================================
# 1. Initialization & Checkpoint Loading
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load global dimensions
metadata = torch.load("global_metadata.pt")
global_max_cell_types = metadata['global_max_cell_types']
global_max_k = metadata['global_max_k']
NUM_CTS_KNOBS = 4 

print("📦 Loading Models for Blind Evaluation...")

# 1A. Load Phase 1 (The Clustering Brain)
phase1_model = FirstTerm(num_cell_types=global_max_cell_types, num_of_clusters=global_max_k).to(device)
phase1_model.load_state_dict(torch.load("phase1_clustering_ep250_twohop.pt", map_location=device))
phase1_model.eval()

# 1B. Load Phase 2 (The Task-Aware MLP)
predictor_model = DirectMLPPredictor(feature_dim=26, num_cts_params=NUM_CTS_KNOBS).to(device)
predictor_model.load_state_dict(torch.load("phase2_predictor.pt", map_location=device))
predictor_model.eval()

# ==========================================
# 2. Test Dataset Setup
# ==========================================
test_csv_file = "./dataset_with_def/unified_manifest_normalized.csv"
processed_dir = "processed_graphs"

# Get all processed graphs to check against the test CSV
all_processed_files = [f for f in os.listdir(processed_dir) if f.endswith('.pt')]

print("\n" + "="*60)
print("🚀 LAUNCHING BLIND TEST ON UNSEEN DESIGNS")
print("="*60)

mae_skew_sum, mae_power_sum, mae_wl_sum = 0, 0, 0
eval_preds = 0
print_limit = 20 # Show the first 20 unseen predictions

with torch.no_grad():
    for filename in all_processed_files:
        placement_id = filename.replace('.pt', '')
        
        # 🚨 This ensures we ONLY evaluate runs that exist in the TEST CSV
        cts_runs = load_cts_parameters(test_csv_file, placement_id, device)
        if not cts_runs: 
            continue # Skip if this design was part of the training set
            
        # Load the graph data for this unseen design
        data = torch.load(os.path.join(processed_dir, filename), map_location=device)

        # 1. Generate the Clustering Matrix (C) for the unseen design
        _, _, _, _, C, X_combined = phase1_model(
            data['X'].to(device), data['X_cell_ids'].to(device), 
            data['num_nodes'], data['p_indices'].to(device), 
            data['A_skip_csr'].to(device), data['current_k'], tau=0.85
        )

        # 2. Evaluate all unseen CTS tool configurations
        for run in cts_runs:
            cts_params = run['knobs'].view(1, -1)
            t_p = run['targets']['power'].view(1, 1)
            t_s = run['targets']['skew'].view(1, 1)
            t_w = run['targets']['wl'].view(1, 1)
            
            # 3. Predict using the MLP
            skew_pred, power_pred, wl_pred = predictor_model(C, X_combined, cts_params)
            
            # Accumulate Absolute Errors
            mae_skew_sum += abs(skew_pred.item() - t_s.item())
            mae_power_sum += abs(power_pred.item() - t_p.item())
            mae_wl_sum += abs(wl_pred.item() - t_w.item())
            
            if eval_preds < print_limit:
                print(f"Unseen {eval_preds+1:2} | "
                      f"SKEW [True: {t_s.item():>6.3f} | Pred: {skew_pred.item():>6.3f}]  ---  "
                      f"POWER [True: {t_p.item():>6.3f} | Pred: {power_pred.item():>6.3f}]")
                
            eval_preds += 1
            
        del C, X_combined, data
        torch.cuda.empty_cache()

# ==========================================
# 3. Final Blind Test Results
# ==========================================
if eval_preds > 0:
    final_mae_skew = mae_skew_sum / eval_preds
    final_mae_power = mae_power_sum / eval_preds
    final_mae_wl = mae_wl_sum / eval_preds

    print("-" * 60)
    print(f"🎯 BLIND TEST GLOBAL MAE (Across {eval_preds} Unseen CTS Runs)")
    print(f"   ➤ Skew MAE:  {final_mae_skew:.4f}")
    print(f"   ➤ Power MAE: {final_mae_power:.4f}")
    print(f"   ➤ WL MAE:    {final_mae_wl:.4f}")
    print("="*60)
else:
    print("⚠️ WARNING: No runs found in the test CSV matching your processed graphs.")

📦 Loading Models for Blind Evaluation...

🚀 LAUNCHING BLIND TEST ON UNSEEN DESIGNS
Unseen  1 | SKEW [True: -1.076 | Pred:  0.743]  ---  POWER [True:  0.725 | Pred:  0.825]
Unseen  2 | SKEW [True: -0.082 | Pred:  1.348]  ---  POWER [True:  0.733 | Pred:  0.542]
Unseen  3 | SKEW [True: -1.198 | Pred: -0.638]  ---  POWER [True:  0.745 | Pred:  0.808]
Unseen  4 | SKEW [True: -0.245 | Pred:  0.129]  ---  POWER [True:  0.967 | Pred:  1.223]
Unseen  5 | SKEW [True: -0.085 | Pred:  0.488]  ---  POWER [True:  0.733 | Pred:  0.832]
Unseen  6 | SKEW [True:  0.553 | Pred:  0.769]  ---  POWER [True:  0.756 | Pred:  0.934]
Unseen  7 | SKEW [True: -0.013 | Pred: -0.300]  ---  POWER [True:  0.902 | Pred:  1.073]
Unseen  8 | SKEW [True: -0.283 | Pred: -0.667]  ---  POWER [True:  0.887 | Pred:  1.334]
Unseen  9 | SKEW [True: -0.237 | Pred: -0.461]  ---  POWER [True:  0.853 | Pred:  1.003]
Unseen 10 | SKEW [True: -0.311 | Pred:  0.215]  ---  POWER [True:  0.803 | Pred:  0.695]
Unseen 11 | SKEW [True:  2.